# Convert Parquet to TSV with chDB

In-process ClickHouse SQL in Python. Run `./generate.sh` first to create `data/events.parquet`.

Requires: `pip install chdb`

In [ ]:
import os
import chdb

os.chdir("data")

## 1. Convert Parquet -> TSV

chDB streams the `SELECT` straight into a TSV file. `TSVWithNames` keeps the column names as a header row.

In [ ]:
chdb.query(
    "SELECT * FROM file('events.parquet') "
    "INTO OUTFILE 'events_chdb.tsv' TRUNCATE FORMAT TSVWithNames"
)

with open("events_chdb.tsv") as f:
    for line in list(f)[:4]:
        print(line.rstrip("\n"))

## 2. Read the TSV back as a DataFrame

The nested `tags` array survives the round-trip: ClickHouse wrote it as `[0,0,0]` in one tab field, and the TSV reader parses it straight back into an array.

In [ ]:
df = chdb.query(
    "SELECT event_id, country, tags FROM file('events_chdb.tsv', 'TSVWithNames') "
    "ORDER BY event_id LIMIT 3",
    "DataFrame",
)
df